# LSTM–GNN с обновляемым графом

Архитектура и обучаемые параметры совпадают со статической LSTM–GNN. Граф пересчитывается для каждого прогноза строго по тому же входному окну длины L. Целевая дата в построении графа не участвует.

Каждый запуск этого ноутбука обрабатывает **все три набора**: синтетику, MOEX и METR-LA.
Наборы загружаются по очереди. Вспомогательные функции находятся в `src/forecast_*.py`.
Результаты каждого опыта и сводный CSV сохраняются автоматически.

Нужны `numpy`, `pandas`, `torch`. Отдельные графовые библиотеки не требуются.

По умолчанию: один seed, CPU, до 15 эпох с ранней остановкой для каждого набора.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from forecast_batch import run_batch
from forecast_experiment import collect_results, fixed_graph
from forecast_models import build_adjacency

print("Проект:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)


Проект: /home/loktar/Desktop/money_coursework
PyTorch: 2.15.0+cpu


## Настройки серии

Каждый ноутбук запускает свою модель на `synthetic`, `moex`, `metr_la` по очереди.
История и горизонт читаются из `schema.json`; текущая постановка — 60 наблюдений и один шаг вперёд.
Используйте одинаковые `DATASETS`, `CONFIG`, `SEEDS`, `RUN_TAG` во всех четырёх ноутбуках.
Один seed — пилот; для оценки разброса позже задайте `[0, 1, 2]` во всех четырёх.

`EVALUATE_TEST=True`: после выбора лучшей эпохи по val рассчитывается test.
Test не участвует в обучении и выборе эпохи. Для новой настройки по val можно задать `False`.
Уже просмотренный test учитываем как исследовательскую оценку при дальнейшей настройке.


In [2]:
MODEL_KIND = "lstm_gnn_dynamic"
DATASETS = ["synthetic", "moex", "metr_la"]
SEEDS = [0]
RUN_TAG = "pilot_v1"
EVALUATE_TEST = True

CONFIG = {
    "hidden_size": 16,
    "batch_size": 32,
    "max_epochs": 15,
    "patience": 4,
    "learning_rate": 0.001,
    "weight_decay": 0.0001,
    "gradient_clip": 1.0,
    "top_k": 3,
    "cpu_threads": 2,
    "device": "cpu",
    "min_delta": 0.0,
}

DATA_ROOT = PROJECT_ROOT / "data/processed"
RESULTS_ROOT = PROJECT_ROOT / "results"
print("Модель:", MODEL_KIND)
print("Наборы:", DATASETS)


Модель: lstm_gnn_dynamic
Наборы: ['synthetic', 'moex', 'metr_la']


## Правило графа и архитектура

Рёбра выбираются top-k по $|\rho_{ij}|$ на входной истории; веса — те же абсолютные корреляции,
нормированные по строке. Диагональ нулевая. $A_{ij}$ означает сообщение от j к i.
Оба графовых варианта используют одинаковую длину истории оценки графа.


\[
h_i=\operatorname{LSTM}_{\theta}(x_{i,t-L+1:t}),\qquad
z_i=\operatorname{ReLU}\!\left(W_s h_i+W_n\sum_j A_{ij}h_j\right),\qquad
\widehat y_i=w^\top z_i+b.
\]

Собственный путь сохраняет информацию вершины даже при отсутствии рёбер.
Граф генератора синтетики моделям не передаётся: связи оцениваются по наблюдаемым значениям.
Динамический вариант меняет и рёбра, и веса; пока результат относится к обновлению графа в целом.
Режим с фиксированными рёбрами и обновляемыми весами предусмотрен в модуле для следующей проверки.

Ниже диагностика вызывается отдельно для каждого набора и использует только train и входные окна val.


## Общий загрузчик и покрытие целей

CSV каждого набора читаются один раз при его запуске. Масштабирование каждого ряда обучается только по наблюдаемым train-значениям.
Пропуски входа заполняются прошлым значением только внутри выбранного окна, затем обучающим средним.
Цели остаются пропущенными; `y_mask` применяется и в функции потерь, и во всех метриках.

Таблица ниже показывает **фактическое число оцениваемых целей** каждой вершины.
Вершина с нулевым покрытием остаётся в составе, но не получает выдуманную нулевую ошибку.
Состав акций по доступности test не пересматривается.

Функция ниже вызывается автоматически внутри цикла по наборам.


In [3]:
def inspect_dataset(data, dataset_name):
    # Keep inspection local so the previous dataset can be released.
    display(pd.DataFrame([
        {"dataset": dataset_name, "split": split, "samples": len(data.dataset(split)),
         "nodes": len(data.nodes), "history": data.history, "features_per_node": 1}
        for split in ("train", "val", "test")
    ]))
    coverage = data.quality().pivot(index="node", columns="split", values="evaluated_targets")
    display(coverage.head(12))
    zero_coverage = coverage.loc[coverage.eq(0).any(axis=1)]
    if len(zero_coverage):
        print("Вершины без оцениваемых целей хотя бы в одном разбиении:")
        display(zero_coverage)
    example = data.dataset("val")[0]
    print("Один вход:", example["X"].shape, "цель:", example["y_norm"].shape)

    static_A, graph_info = fixed_graph(data, CONFIG["top_k"])
    print("Фиксированный граф:", graph_info)
    display(pd.DataFrame(static_A.numpy(), index=data.nodes, columns=data.nodes).iloc[:6, :6])
    diagnostics = []
    validation = data.dataset("val")
    for index in sorted({0, len(validation) // 2, len(validation) - 1}):
        sample = validation[index]
        dynamic_A = build_adjacency(torch.from_numpy(sample["X"]), CONFIG["top_k"])
        fixed_edges, current_edges = static_A > 0, dynamic_A > 0
        union = int((fixed_edges | current_edges).sum())
        intersection = int((fixed_edges & current_edges).sum())
        diagnostics.append({
            "origin_time": str(data.times[int(sample["origin_row"])]),
            "edge_jaccard_vs_static": intersection / union if union else 1.0,
            "mean_absolute_weight_change": float((static_A - dynamic_A).abs().mean()),
        })
    display(pd.DataFrame(diagnostics))


## Запуск на трёх наборах

Нейросети обучаются по MSE нормированных целей с маской. Лучший checkpoint выбирается только по val:
по MAE для синтетики и METR-LA, по QLIKE для MOEX. У всех нейросетей одинаковые правила оптимизации,
ранней остановки и перемешивания обучающих примеров.

MAE/RMSE считаются в исходной шкале. Для MOEX прогноз log(RV) переводится через exp в RV;
QLIKE используется в форме `log(v_hat) + RV/v_hat`. Её значение может быть отрицательным;
сравнивать его можно на одинаковых наблюдениях.

Повторный запуск той же конфигурации переиспользует проверенный результат.
Для повторения с другой инициализацией меняйте seed. `RUN_TAG` помогает разделять серии опытов.

После каждого завершённого опыта обновляется `summary.csv` этой серии.
В `status.csv` виден результат каждого набора и seed; `errors.csv` содержит ошибки.
При ошибке следующий набор всё равно запускается; в конце частичная серия явно отмечается.
Повторный запуск переиспользует **завершённые** опыты с той же конфигурацией.
Прерванное обучение текущий runner не возобновляет: для него потребуется новый `RUN_TAG`.


In [4]:
batch = run_batch(
    DATA_ROOT, MODEL_KIND, RESULTS_ROOT,
    datasets=DATASETS, seeds=SEEDS, config=CONFIG,
    evaluate_test=EVALUATE_TEST, run_tag=RUN_TAG,
    inspect_dataset=inspect_dataset,
)
reports = batch["reports"]
summary = batch["summary"]
SUMMARY_DIR = batch["summary_dir"]
display(batch["status"])
display(summary)
print("Сводные результаты:", SUMMARY_DIR / "summary.csv")



=== lstm_gnn_dynamic: synthetic ===


,dataset,split,samples,nodes,history,features_per_node
0,synthetic,train,4140,12,60,1
1,synthetic,val,600,12,60,1
2,synthetic,test,1201,12,60,1


split,test,train,val
node,,,
node_0,1201,4140,600
node_1,1201,4140,600
node_10,1201,4140,600
node_11,1201,4140,600
node_2,1201,4140,600
node_3,1201,4140,600
node_4,1201,4140,600
node_5,1201,4140,600
node_6,1201,4140,600


Один вход: (60, 12, 1) цель: (12,)
Фиксированный граф: {'anchor_origin_row': 4199, 'anchor_time': '4199', 'history': 60, 'rule': 'row-normalized top-k absolute Pearson correlation', 'A_ij': 'message from node j to node i', 'self_edges': False, 'estimated_on': 'last history-length window within train only'}


,node_0,node_1,node_2,node_3,node_4,node_5
node_0,0.000000,0.000000,0.420999,0.000000,0.0,0.000000
node_1,0.000000,0.000000,0.000000,0.313441,0.0,0.345814
node_2,0.427015,0.000000,0.000000,0.000000,0.0,0.000000
node_3,0.000000,0.364898,0.000000,0.000000,0.0,0.405260
node_4,0.265694,0.000000,0.000000,0.426528,0.0,0.307778
node_5,0.000000,0.000000,0.000000,0.000000,0.0,0.000000


,origin_time,edge_jaccard_vs_static,mean_absolute_weight_change
0,4199,1.000000,0.000000
1,4499,0.142857,0.127282
2,4798,0.180328,0.119343


... (14 outputs left out)

lstm_gnn_dynamic seed=0 epoch=1: train MSE=0.32149; val mae=2.46398
lstm_gnn_dynamic seed=0 epoch=2: train MSE=0.246; val mae=2.42407
lstm_gnn_dynamic seed=0 epoch=3: train MSE=0.24318; val mae=2.38925
lstm_gnn_dynamic seed=0 epoch=4: train MSE=0.24015; val mae=2.36656
lstm_gnn_dynamic seed=0 epoch=5: train MSE=0.23708; val mae=2.38912
lstm_gnn_dynamic seed=0 epoch=6: train MSE=0.2344; val mae=2.3946
lstm_gnn_dynamic seed=0 epoch=7: train MSE=0.23183; val mae=2.32985
lstm_gnn_dynamic seed=0 epoch=8: train MSE=0.23016; val mae=2.3231
lstm_gnn_dynamic seed=0 epoch=9: train MSE=0.22954; val mae=2.32073
lstm_gnn_dynamic seed=0 epoch=10: train MSE=0.22886; val mae=2.3074
lstm_gnn_dynamic seed=0 epoch=11: train MSE=0.22846; val mae=2.33377
lstm_gnn_dynamic seed=0 epoch=12: train MSE=0.22828; val mae=2.30517
lstm_gnn_dynamic seed=0 epoch=13: train MSE=0.2281; val mae=2.31971
lstm_gnn_dynamic seed=0 epoch=14: train MSE=0.22789; val mae=2.28861
lstm_gnn_dynamic seed=0 epoch=15: train MSE=0.2275

,dataset,model,seed,status,error,output_dir
0,synthetic,lstm_gnn_dynamic,0,completed,,/home/loktar/Desktop/money_coursework/results/...
1,moex,lstm_gnn_dynamic,0,completed,,/home/loktar/Desktop/money_coursework/results/...
2,metr_la,lstm_gnn_dynamic,0,completed,,/home/loktar/Desktop/money_coursework/results/...


,dataset,model,seed,split,primary_metric,primary_score,mae,rmse,qlike,normalized_mse,n_observed,n_origins,best_epoch,parameters,fit_seconds,comparison_id,run_id,output_dir
0,synthetic,lstm_gnn_dynamic,0,val,mae,0.082079,0.082079,0.102521,NaN,0.811078,7200,600,4,1761,9.712717,36989cce73a0,dde4f29fcf43,/home/loktar/Desktop/money_coursework/results/...
1,synthetic,lstm_gnn_dynamic,0,test,mae,0.081912,0.081912,0.102494,NaN,0.810703,14412,1201,4,1761,9.712717,36989cce73a0,dde4f29fcf43,/home/loktar/Desktop/money_coursework/results/...
2,moex,lstm_gnn_dynamic,0,val,qlike,-7.468197,0.000134,0.000289,-7.468197,0.521479,2371,252,14,1761,3.864365,362a12d77352,47d20d15c6fa,/home/loktar/Desktop/money_coursework/results/...
3,moex,lstm_gnn_dynamic,0,test,qlike,-7.283683,0.000147,0.000582,-7.283683,0.490024,2283,254,14,1761,3.864365,362a12d77352,47d20d15c6fa,/home/loktar/Desktop/money_coursework/results/...
4,metr_la,lstm_gnn_dynamic,0,val,mae,2.288611,2.288611,3.941921,NaN,0.226127,660000,3246,14,1761,3277.771814,c43da30bee45,ba4961e8f604,/home/loktar/Desktop/money_coursework/results/...
5,metr_la,lstm_gnn_dynamic,0,test,mae,2.472050,2.472050,4.204316,NaN,0.244935,1247252,6189,14,1761,3277.771814,c43da30bee45,ba4961e8f604,/home/loktar/Desktop/money_coursework/results/...


Сводные результаты: /home/loktar/Desktop/money_coursework/results/summaries/lstm_gnn_dynamic/20261124T102059_345487Z/summary.csv


## Сохранённые результаты и примеры

Подробные результаты: `results/<dataset>/<model>/seed_<seed>/<run_id>/`.
Там находятся `run.json`, метрики, ошибки по датам `*_losses.csv`, метрики по вершинам,
прогнозы с исходной маской в `*_predictions.npz`. У нейросетей также сохраняются история обучения и лучшая модель.

Сводка этой серии: `results/summaries/<model>/<дата_UTC>/summary.csv`.
Одна строка — один набор, seed и разбиение val/test; при одном seed и трёх успешных наборах получится 6 строк.
В сводке есть основная метрика, MAE, RMSE, QLIKE для MOEX, число целей, время обучения и путь к опыту.
Разные серии получают отдельные папки. Ниже показаны примеры для каждого успешно обработанного набора.


In [5]:
preview_split = "test" if EVALUATE_TEST else "val"
for dataset_name in DATASETS:
    dataset_reports = [report for report in reports if report["dataset"] == dataset_name]
    if not dataset_reports:
        continue
    run_dir = Path(dataset_reports[0]["output_dir"])
    print(dataset_name, "—", run_dir)
    display(pd.read_csv(run_dir / f"{preview_split}_per_node.csv").head(12))
    with np.load(run_dir / f"{preview_split}_predictions.npz") as saved:
        preview = pd.DataFrame({
            "node": saved["node_order"], "observed": saved["mask"][0],
            "actual_raw": saved["target_raw"][0], "prediction_raw": saved["prediction_raw"][0],
        })
    display(preview.head(12))


synthetic — /home/loktar/Desktop/money_coursework/results/synthetic/lstm_gnn_dynamic/seed_000/dde4f29fcf43


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,node_0,node_0,1201,1201,0.082588,0.104351
1,1,node_1,node_1,1201,1201,0.078562,0.097782
2,2,node_2,node_2,1201,1201,0.082693,0.103647
3,3,node_3,node_3,1201,1201,0.082594,0.101593
4,4,node_4,node_4,1201,1201,0.080772,0.102015
5,5,node_5,node_5,1201,1201,0.080623,0.100837
6,6,node_6,node_6,1201,1201,0.083928,0.106018
7,7,node_7,node_7,1201,1201,0.081970,0.101541
8,8,node_8,node_8,1201,1201,0.080448,0.101670
9,9,node_9,node_9,1201,1201,0.084369,0.104003


,node,observed,actual_raw,prediction_raw
0,node_0,True,-0.018677,-0.006454
1,node_1,True,0.002347,0.017020
2,node_2,True,-0.135561,-0.005512
3,node_3,True,0.063401,0.036604
4,node_4,True,0.048733,-0.018571
5,node_5,True,-0.123078,-0.036499
6,node_6,True,-0.013832,-0.057522
7,node_7,True,-0.040185,0.005925
8,node_8,True,0.174519,0.057142
9,node_9,True,-0.034389,-0.016158


moex — /home/loktar/Desktop/money_coursework/results/moex/lstm_gnn_dynamic/seed_000/47d20d15c6fa


,node_id,node,name,observed_pairs,n_observed,mae,rmse,qlike
0,0,SBER,SBER,254,254,0.000077,0.000148,-8.048693
1,1,GAZP,GAZP,254,254,0.000169,0.000322,-7.096461
2,2,LKOH,LKOH,254,254,0.000101,0.000205,-7.623963
3,3,GMKN,GMKN,254,254,0.000125,0.000194,-7.125086
4,4,YNDX,YNDX,0,0,NaN,NaN,NaN
5,5,ROSN,ROSN,254,254,0.000114,0.000189,-7.403775
6,6,PLZL,PLZL,251,251,0.000141,0.000264,-7.188991
7,7,MGNT,MGNT,254,254,0.000141,0.000247,-7.307587
8,8,NVTK,NVTK,254,254,0.000193,0.000351,-6.854705
9,9,VTBR,VTBR,254,254,0.000264,0.001598,-6.902772


,node,observed,actual_raw,prediction_raw
0,SBER,True,0.000196,0.000142
1,GAZP,True,0.000641,0.000281
2,LKOH,True,0.000351,0.000154
3,GMKN,True,0.000510,0.000265
4,YNDX,False,NaN,0.000342
5,ROSN,True,0.000478,0.000207
6,PLZL,True,0.000058,0.000117
7,MGNT,True,0.000848,0.000454
8,NVTK,True,0.000861,0.000334
9,VTBR,True,0.000401,0.000252


metr_la — /home/loktar/Desktop/money_coursework/results/metr_la/lstm_gnn_dynamic/seed_000/ba4961e8f604


,node_id,node,name,observed_pairs,n_observed,mae,rmse
0,0,773869,773869,6188,6188,1.986885,3.887239
1,1,767541,767541,6189,6189,1.555876,2.461760
2,2,767542,767542,6189,6189,1.803446,3.225036
3,3,717447,717447,6185,6185,2.523483,3.986323
4,4,717446,717446,5963,5963,3.560871,5.043768
5,5,717445,717445,6185,6185,3.330783,5.071400
6,6,773062,773062,6185,6185,2.503248,3.974914
7,7,767620,767620,6186,6186,1.661730,2.435929
8,8,737529,737529,6088,6088,2.384217,3.904823
9,9,717816,717816,6186,6186,2.632198,4.495320


,node,observed,actual_raw,prediction_raw
0,773869,True,64.750,64.367024
1,767541,True,56.375,61.029515
2,767542,True,69.375,66.933286
3,717447,True,58.000,57.995133
4,717446,True,66.000,64.948563
5,717445,True,44.375,53.115918
6,773062,True,63.125,64.776903
7,767620,True,52.250,62.317500
8,737529,True,65.750,61.079506
9,717816,True,68.000,66.906402


## Сопоставление четырёх запусков

Сравнивайте строки с одним `comparison_id`: он фиксирует данные, нормировку, общие настройки и версии кода.
Метрики разных наборов данных имеют разные единицы и напрямую не сопоставляются.

Обычная LSTM здесь многомерная, а LSTM–GNN использует общий по вершинам временной кодировщик.
Поэтому их сравнение относится к двум семействам моделей, а не к изолированному добавлению одного слоя.
Главная проверка обновления графа — **static против dynamic** при одинаковых seed и настройках.

Текущая синтетика со статическим истинным графом служит контрольным сценарием.
Для проверки пользы изменения топологии позже потребуется синтетика с меняющимися связями.

`comparison.csv` собирает уже завершённые запуски других моделей с теми же протоколом и seed
внутри каждого набора. После запуска всех четырёх ноутбуков таблица последнего включает все четыре модели.
При одном seed и наличии val/test это 24 строки. Ошибки разных наборов не усредняются.


In [6]:
comparison_parts = []
for dataset_name in DATASETS:
    current_ids = summary.loc[summary["dataset"].eq(dataset_name), "comparison_id"].unique()
    for split in (("val", "test") if EVALUATE_TEST else ("val",)):
        table = collect_results(RESULTS_ROOT, dataset_name, split=split, run_tag=RUN_TAG)
        if table.empty:
            continue
        table = table.loc[table["comparison_id"].isin(current_ids) & table["seed"].isin(SEEDS)].copy()
        if not table.empty:
            table.insert(0, "dataset", dataset_name)
            comparison_parts.append(table)
comparison = pd.concat(comparison_parts, ignore_index=True) if comparison_parts else pd.DataFrame(
    columns=["dataset", "model", "seed", "split", "comparison_id", "mae", "rmse", "qlike"]
)
comparison.to_csv(SUMMARY_DIR / "comparison.csv", index=False)
display(comparison)
print("Сопоставимые запуски:", SUMMARY_DIR / "comparison.csv")

failed = batch["status"].loc[batch["status"]["status"].ne("completed")]
if not failed.empty:
    display(failed)
    raise RuntimeError(f"Серия завершена частично. Ошибки: {SUMMARY_DIR / 'errors.csv'}")
print("Готово: результаты сохранены для всех трёх наборов.")


,dataset,model,seed,split,comparison_id,mae,rmse,n_observed,best_epoch,parameters,fit_seconds,output_dir,qlike
0,synthetic,lstm,0,val,36989cce73a0,0.081100,0.101304,7200,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
1,synthetic,lstm_gnn_dynamic,0,val,36989cce73a0,0.082079,0.102521,7200,4.0,1761,9.712717,/home/loktar/Desktop/money_coursework/results/...,NaN
2,synthetic,lstm_gnn_static,0,val,36989cce73a0,0.081928,0.102395,7200,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
3,synthetic,persistence,0,val,36989cce73a0,0.096973,0.121423,7200,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
4,synthetic,lstm,0,test,36989cce73a0,0.080639,0.101053,14412,14.0,2124,11.641013,/home/loktar/Desktop/money_coursework/results/...,NaN
5,synthetic,lstm_gnn_dynamic,0,test,36989cce73a0,0.081912,0.102494,14412,4.0,1761,9.712717,/home/loktar/Desktop/money_coursework/results/...,NaN
6,synthetic,lstm_gnn_static,0,test,36989cce73a0,0.081709,0.102267,14412,4.0,1761,17.240769,/home/loktar/Desktop/money_coursework/results/...,NaN
7,synthetic,persistence,0,test,36989cce73a0,0.096982,0.121153,14412,NaN,0,0.000000,/home/loktar/Desktop/money_coursework/results/...,NaN
8,moex,lstm,0,val,362a12d77352,0.000145,0.000304,2371,15.0,1962,2.750675,/home/loktar/Desktop/money_coursework/results/...,-7.398339
9,moex,lstm_gnn_dynamic,0,val,362a12d77352,0.000134,0.000289,2371,14.0,1761,3.864365,/home/loktar/Desktop/money_coursework/results/...,-7.468197


Сопоставимые запуски: /home/loktar/Desktop/money_coursework/results/summaries/lstm_gnn_dynamic/20261124T102059_345487Z/comparison.csv
Готово: результаты сохранены для всех трёх наборов.
